In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U transformers
!pip install -q langchain langchain-community langchain-core
!pip install -q langchain-huggingface
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q langchain-text-splitters
!pip install -q fastapi uvicorn pyngrok accelerate bitsandbytes requests

ERROR: Could not find a version that satisfies the requirement langchain-community (from versions: none)
ERROR: No matching distribution found for langchain-community
ERROR: Could not find a version that satisfies the requirement langchain-huggingface (from versions: none)
ERROR: No matching distribution found for langchain-huggingface
ERROR: Could not find a version that satisfies the requirement faiss-cpu (from versions: none)
ERROR: No matching distribution found for faiss-cpu
ERROR: Could not find a version that satisfies the requirement langchain-text-splitters (from versions: none)
ERROR: No matching distribution found for langchain-text-splitters


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import (
    StructuredOutputParser,
    ResponseSchema
)

import torch
import re
import os

In [ ]:
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 3

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model device:", model.device)

In [ ]:
prompt = """
Explain what a software project contract is in two sentences.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.95
    )

generated_tokens = outputs[0][
    inputs["input_ids"].shape[1]:
]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import torch

app = FastAPI(
    title="AI Document Intelligence API"
)


class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 300


class GenerateResponse(BaseModel):
    response: str


@app.get("/")
def root():
    return {
        "status": "running",
        "message": "Mistral-Nemo API is running"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post(
    "/generate",
    response_model=GenerateResponse
)
def generate(request: GenerateRequest):

    inputs = tokenizer(
        request.prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "response": response
    }

In [ ]:
import threading
import uvicorn


def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("API server started on port 8000")

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(
    "Token"
)

In [ ]:
tunnel = ngrok.connect(8000, "http")

print("Tunnel object:")
print(tunnel)

print("\nPublic URL:")
print(tunnel.public_url)

In [ ]:
import requests

url = tunnel.public_url + "/generate"

print("Testing:")
print(url)

response = requests.post(
    url,
    json={
        "prompt": "Explain what RAG is in two sentences.",
        "max_new_tokens": 100
    },
    timeout=300
)

print("Status:", response.status_code)
print("Response:", response.json())

In [ ]:
def generate_text(
    prompt,
    max_new_tokens=300
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
def load_pdf(pdf_path):

    if not os.path.isfile(pdf_path):
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    if not documents:
        raise ValueError(
            "No content could be extracted from the PDF."
        )

    if not any(
        doc.page_content.strip()
        for doc in documents
    ):
        raise ValueError(
            "The PDF contains no extractable text."
        )

    return documents

In [ ]:
pdf_path = "pdf_path"

In [ ]:
documents = load_pdf(pdf_path)

print("Number of pages:", len(documents))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

In [ ]:
vectordb = FAISS.from_documents(
    chunks,
    embedding_model
)

In [ ]:
print("FAISS vector database created.")

In [ ]:
test_question = "What are the payment conditions?"

retrieved_docs = vectordb.similarity_search(
    test_question,
    k=TOP_K
)

for doc in retrieved_docs:
    print(doc.metadata)

In [ ]:
def build_context(docs):
    context_parts = []

    for doc in docs:
        page = doc.metadata.get("page", "Unknown")
        page_number = page + 1 if isinstance(page, int) else page

        context_parts.append(
            f"[Page {page_number}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(context_parts)

In [ ]:
context = build_context(retrieved_docs)

In [ ]:
qa_template = """
You are a document analysis assistant.

Answer the question using ONLY the provided context.

IMPORTANT RULES:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer cannot be found in the context,
   say that the information is not available in the document.
4. Do NOT invent page numbers.
5. If the question asks for page numbers, use ONLY the
   [Page X] labels provided in the context.
6. If the question asks for a number, date, amount, name,
   or condition, preserve it exactly as stated in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
def answer_question(vectordb, question, k=TOP_K):

    docs = vectordb.similarity_search(
        question,
        k=k
    )

    context = build_context(docs)

    prompt = qa_template.format(
        context=context,
        question=question
    )

    answer = generate_text(
        prompt,
        max_new_tokens=300
    )

    source_pages = sorted(set(
        doc.metadata["page"] + 1
        for doc in docs
        if isinstance(doc.metadata.get("page"), int)
    ))

    return {
        "answer": answer,
        "sources": docs,
        "source_pages": source_pages
    }

In [ ]:
test_question = "What are the payment conditions?"

result = answer_question(
    vectordb,
    test_question
)

print(result["answer"])

print("\nSources:")
for doc in result["sources"]:
    print(doc.metadata)

In [ ]:
def summarize_chunks(chunks):
    summaries = []

    for i, chunk in enumerate(chunks):

        prompt = f"""
You are a document summarization assistant.

Summarize the following document section.
Keep the important facts, numbers, names,
dates, and technical information.

Document section:
{chunk.page_content}

Summary:
"""

        summary = generate_text(
            prompt,
            max_new_tokens=200
        )

        summaries.append(summary)

    return summaries

In [ ]:
def summarize_document(chunks):

    chunk_summaries = summarize_chunks(chunks)

    combined_summary = "\n\n".join(
        chunk_summaries
    )

    prompt = f"""
You are an intelligent document analysis assistant.

Create a final summary from the following
section summaries.

Provide:
1. Concise overall summary
2. Important key points
3. Potential risks

Section summaries:
{combined_summary}

Final summary:
"""

    return generate_text(
        prompt,
        max_new_tokens=500
    )

In [ ]:
summary = summarize_document(chunks)

print(summary)

In [ ]:
document_schema = ResponseSchema(
    name="document",
    description="Name or description of the analyzed document."
)

summary_schema = ResponseSchema(
    name="summary",
    description="Concise summary of the document."
)

key_points_schema = ResponseSchema(
    name="key_points",
    description="Important points extracted from the document."
)

risks_schema = ResponseSchema(
    name="risks",
    description="Potential risks or concerns identified in the document."
)

references_schema = ResponseSchema(
    name="references",
    description="Relevant page numbers or document sections."
)

In [ ]:
response_schemas = [
    document_schema,
    summary_schema,
    key_points_schema,
    risks_schema,
    references_schema
]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

In [ ]:
format_instructions = (
    output_parser.get_format_instructions()
)

In [ ]:
analysis_template = """
You are an intelligent document analysis assistant.

Analyze the provided document context.

Use ONLY the information contained in the context.

Do not invent information.

Context:
{context}

{format_instructions}

Return ONLY the structured output.
"""

In [ ]:
analysis_prompt = PromptTemplate(
    template=analysis_template,
    input_variables=[
        "context",
        "format_instructions"
    ]
)

In [ ]:
def extract_json_block(text):

    match = re.search(
        r"```json\s*(.*?)\s*```",
        text,
        re.DOTALL
    )

    if match:
        return match.group(1)

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if match:
        return match.group(0)

    raise ValueError(
        "No JSON object found in model response."
    )

In [ ]:
def analyze_document(
    pdf_path,
    operation="qa",
    question=None
):

    documents = load_pdf(pdf_path)

    chunks = text_splitter.split_documents(
        documents
    )

    vectordb = FAISS.from_documents(
        chunks,
        embedding_model
    )

    if operation == "qa":

        if not question:
            raise ValueError(
                "Question is required for Q&A."
            )

        result = answer_question(
            vectordb,
            question
        )

        return result

    elif operation == "summary":

        summary = summarize_document(
            chunks
        )

        return summary

    elif operation == "analysis":

        context = build_context(chunks)

        prompt = analysis_template.format(
            context=context,
            format_instructions=format_instructions
        )

        response = generate_text(
            prompt,
            max_new_tokens=500
        )

        json_text = extract_json_block(
            response
        )

        return output_parser.parse(
            json_text
        )

    else:

        raise ValueError(
            f"Unsupported operation: {operation}"
        )

In [ ]:
def prepare_document(pdf_path):

    documents = load_pdf(pdf_path)

    chunks = text_splitter.split_documents(
        documents
    )

    summary = summarize_document(chunks)

    return {
        "documents": documents,
        "chunks": chunks,
        "summary": summary
    }

In [ ]:
comparison_template = """
You are a precise document comparison assistant.

Your task is to compare Document A and Document B using ONLY
the exact information provided in the two documents.

IMPORTANT RULES:

1. Do not invent information.

2. Do not use outside knowledge.

3. Do not assume that information was removed just because it
   is summarized differently.

4. Only report something as ADDED if it appears in Document B
   and does not appear in Document A.

5. Only report something as REMOVED if it appears in Document A
   and does not appear in Document B.

6. If a value, date, amount, duration, number, or condition
   changes, explicitly show BOTH the old value and the new value.

   Example:
   - Project fee: USD 12,000 → USD 15,000
   - Support period: 30 days → 60 days
   - Payment installments: 4 → 3

7. Preserve exact numbers, dates, names, technical terms,
   and conditions from the documents.

8. Do not describe a specific change only in general terms.
   Give the exact old and new values whenever they are available.

9. If something is uncertain or cannot be determined from the
   documents, write:
   "Not clearly determined from the documents."

10. Do not infer that a feature was removed unless the documents
    clearly support that conclusion.

11. If a feature or requirement appears in both documents but
    is described differently, treat it as a difference rather
    than automatically marking it as removed.

12. Separate actual document changes from potential risks.

13. The "risks" section should contain only risks that can
    reasonably result from the actual changes between the documents.

DOCUMENT A:
{document_a}

DOCUMENT B:
{document_b}

Compare the documents and identify:

A. OVERALL COMPARISON
Give a concise description of the overall changes.

B. DIFFERENCES
List all important changed values, dates, amounts, durations,
requirements, conditions, or features.

For numerical or factual changes, use this format:

- Item: OLD VALUE → NEW VALUE

C. ADDED
List information that exists in Document B but not in Document A.

D. REMOVED
List information that exists in Document A but not in Document B.

If nothing was removed, write:
"No information was removed."

E. RISKS
List potential risks caused by the actual changes between the
two documents.

Do not invent risks that are unrelated to the changes.

{format_instructions}

Return ONLY the structured output.
"""

In [ ]:
comparison_document_schema = ResponseSchema(
    name="comparison",
    description="Overall comparison between Document A and Document B."
)

differences_schema = ResponseSchema(
    name="differences",
    description="Important differences between the two documents."
)

added_schema = ResponseSchema(
    name="added",
    description="Information present in Document B but not Document A."
)

removed_schema = ResponseSchema(
    name="removed",
    description="Information present in Document A but not Document B."
)

risks_schema = ResponseSchema(
    name="risks",
    description="Potential risks or impacts caused by the changes."
)

In [ ]:
comparison_schemas = [
    comparison_document_schema,
    differences_schema,
    added_schema,
    removed_schema,
    risks_schema
]

comparison_parser = StructuredOutputParser.from_response_schemas(
    comparison_schemas
)

comparison_format_instructions = (
    comparison_parser.get_format_instructions()
)

In [ ]:
def compare_documents(pdf_path_a, pdf_path_b):

    document_a = prepare_document(pdf_path_a)
    document_b = prepare_document(pdf_path_b)

    document_a_text = "\n\n".join(
        chunk.page_content
        for chunk in document_a["chunks"]
    )

    document_b_text = "\n\n".join(
        chunk.page_content
        for chunk in document_b["chunks"]
    )

    prompt = comparison_template.format(
        document_a=document_a_text,
        document_b=document_b_text,
        format_instructions=comparison_format_instructions
    )

    response = generate_text(
        prompt,
        max_new_tokens=700
    )

    json_text = extract_json_block(response)

    result = comparison_parser.parse(
        json_text
    )

    return result

In [ ]:
pdf_path_a = "pdf_path_a"

pdf_path_b = "pdf_path_b"

comparison_result = compare_documents(
    pdf_path_a,
    pdf_path_b
)

print(comparison_result)

In [ ]:
qa_result = analyze_document(
    pdf_path,
    operation="qa",
    question="On which pages are the payment conditions mentioned?"
)

print(qa_result["answer"])

In [ ]:
summary_result = analyze_document(
    pdf_path,
    operation="summary"
)

print(summary_result)

In [ ]:
analysis_result = analyze_document(
    pdf_path,
    operation="analysis"
)

print(analysis_result)